
# Matryoshka embeddings on Colab — MRL / ESE / MIPIC

Clone [`duncan-nguyen/embedding-mrl`](https://github.com/duncan-nguyen/embedding-mrl),
pick a method and a backbone in the form below, and run one experiment end to end.

**How to use**
1. `Runtime → Change runtime type → GPU` (T4 is enough for BERT / TinyBERT).
2. Set everything in **§1 Experiment settings** — nothing below needs editing.
3. `Runtime → Run all`.

Training data and every evaluation CSV ship inside the repo, so the only
download at runtime is the backbone from the Hugging Face Hub.


## 1. Experiment settings

In [ ]:

#@title  Experiment settings { display-mode: "form" }

#@markdown ### Method and backbone
METHOD = "mipic"  #@param ["mrl", "ese", "mipic"]
BACKBONE = "bert"  #@param ["bert", "tinybert_6l", "bgem3", "qwen3_0.6b"]
#@markdown `bert` (110M) and `tinybert_6l` (67M) fit a free T4 comfortably.
#@markdown `bgem3` (568M) and `qwen3_0.6b` (596M) need a smaller batch, and
#@markdown Qwen3 also needs `trust_remote_code`, which the config already sets.

#@markdown ### Training  &nbsp; *(Table 7 defaults: 5 epochs, lr 2e-5, batch 64, len 256)*
EPOCHS = 5  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
MAX_LENGTH = 256  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
FP16 = True  #@param {type:"boolean"}
MAX_GRAD_NORM = 1.0  #@param {type:"number"}
#@markdown `MAX_TRAIN_SAMPLES = 0` uses the full 20 244-sentence corpus.
#@markdown Set it to e.g. 2000 for a quick smoke run first.
MAX_TRAIN_SAMPLES = 0  #@param {type:"integer"}

#@markdown ### Evaluation
EVAL_SPLIT = "test"  #@param ["test", "validation"]
#@markdown The full suite is slow; leave `EVAL_EVERY_EPOCH` off to score once at the end.
EVAL_EVERY_EPOCH = False  #@param {type:"boolean"}

#@markdown ### Repository and output
REPO_URL = "https://github.com/duncan-nguyen/embedding-mrl"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
RUN_NAME = ""  #@param {type:"string"}
#@markdown Copy the finished run to `MyDrive/embedding-mrl/` so it survives the session.
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

RUN_NAME = RUN_NAME or f"{METHOD}_{BACKBONE}"
print(f"{METHOD.upper()} on {BACKBONE} -> run '{RUN_NAME}'")
print(f"{EPOCHS} epochs, batch {BATCH_SIZE}, lr {LEARNING_RATE}, max_length {MAX_LENGTH}")


## 2. Runtime check

In [ ]:

import subprocess, sys

print("Python", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU   ", gpu)
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu = ""
    print("GPU    none — this will run on CPU and take hours.")
    print("       Runtime -> Change runtime type -> T4 GPU, then re-run.")

# The two large backbones do not fit a 16GB card at batch 16 / length 256.
if BACKBONE in ("bgem3", "qwen3_0.6b") and "16" in gpu.split(",")[-1]:
    print(f"\nNote: {BACKBONE} on a 16GB card usually needs BATCH_SIZE 4-8 "
          f"(currently {BATCH_SIZE}) or MAX_LENGTH 128.")

## 3. Clone the repository and install dependencies

In [ ]:


import os, subprocess, sys

from pathlib import Path



REPO_DIR = Path("/content/embedding-mrl") if Path("/content").exists() else Path.cwd() / "embedding-mrl"



def run(cmd, **kwargs):

    """Run a command, streaming its output so long steps are not silent."""

    process = subprocess.Popen(

        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,

        text=True, bufsize=1, **kwargs,

    )

    for line in process.stdout:

        print(line, end="")

    process.wait()

    return process.returncode



if not REPO_DIR.exists():

    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

else:

    print(f"{REPO_DIR} already present — pulling instead")

    run(["git", "pull", "--ff-only"], cwd=str(REPO_DIR))



os.chdir(REPO_DIR)

print("\nworking directory:", Path.cwd())

print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)



# Colab already ships torch, pandas, sklearn and scipy; this fills the gaps.

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib"])

## 4. Build the run configuration

Everything from §1 becomes a `--set key=value` override on top of the shipped
YAML, so the file on disk is never edited and the resolved config is printed
before anything trains.

In [ ]:

from pathlib import Path

CONFIG_PATH = Path("configs") / METHOD / f"{BACKBONE}.yaml"
OUTPUT_DIR = Path("outputs") / RUN_NAME

if not CONFIG_PATH.exists():
    available = sorted(str(p) for p in Path("configs").rglob("*.yaml") if p.name != "base.yaml")
    raise FileNotFoundError(
        f"{CONFIG_PATH} is not on branch '{BRANCH}'.\nAvailable:\n  " + "\n  ".join(available)
    )

def yaml_value(value):
    """Render a Python value as a literal PyYAML reads back as the same type.

    PyYAML is YAML 1.1, whose float resolver needs *both* a decimal point and a
    signed exponent - so a bare `2e-05` comes back as the string "2e-05" and
    lands in `train.lr`. Emit `2.0e-05` instead.
    """
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None or value == "":
        return "null"
    if isinstance(value, float):
        text = repr(value)
        if "e" in text:
            mantissa, _, exponent = text.partition("e")
            if "." not in mantissa:
                mantissa += ".0"
            if not exponent.startswith(("+", "-")):
                exponent = "+" + exponent
            text = f"{mantissa}e{exponent}"
        return text
    return str(value)

overrides = [
    f"name={RUN_NAME}",
    f"train.output_dir={OUTPUT_DIR}",
    f"train.epochs={EPOCHS}",
    f"train.batch_size={BATCH_SIZE}",
    f"train.lr={yaml_value(float(LEARNING_RATE))}",
    f"train.seed={SEED}",
    f"train.fp16={yaml_value(FP16)}",
    f"train.max_grad_norm={yaml_value(MAX_GRAD_NORM if MAX_GRAD_NORM > 0 else None)}",
    f"data.max_length={MAX_LENGTH}",
    f"eval.split={EVAL_SPLIT}",
    f"eval.every_epoch={yaml_value(EVAL_EVERY_EPOCH)}",
]
overrides.append("data.train_file=train/merged_all.csv")

if MAX_TRAIN_SAMPLES > 0:
    overrides.append(f"data.max_train_samples={MAX_TRAIN_SAMPLES}")

TRAIN_ARGS = [str(a) for pair in (("--set", o) for o in overrides) for a in pair]
print(f"config: {CONFIG_PATH}\noverrides:")
for override in overrides:
    print("   ", override)

print("\n" + "=" * 60 + "\nresolved configuration\n" + "=" * 60)
run([sys.executable, "scripts/train.py", "--config", str(CONFIG_PATH), *TRAIN_ARGS, "--print-config"])

## 5. Train

The backbone downloads on first use. With BERT-base, the full corpus and the
Table 7 defaults this is roughly 1 265 steps per epoch.

In [ ]:

import time

started = time.time()
exit_code = run([sys.executable, "scripts/train.py", "--config", str(CONFIG_PATH), *TRAIN_ARGS])
elapsed = time.time() - started

print(f"\nfinished in {elapsed / 60:.1f} min (exit code {exit_code})")
if exit_code != 0:
    raise RuntimeError("training failed — see the log above")

## 6. Results

In [ ]:

import json
import pandas as pd

report = json.loads((OUTPUT_DIR / "results.json").read_text())

print(f"{report['experiment']['name']}  ({report['experiment']['method']}, "
      f"{report['experiment']['model']})")
training = report["training"]
loss = training["final_loss"]
print(f"trained {training['epochs_completed']} epochs, "
      f"final loss {'n/a' if loss is None else format(loss, '.4f')}, "
      f"{training['duration_seconds'] or 0:.0f}s")

table = pd.read_csv(OUTPUT_DIR / "results.csv").set_index("dim")
pd.set_option("display.width", 200, "display.max_columns", 50)
table

## 7. Quality against the storage rate

The whole point of a Matryoshka model is the shape of these curves, not the
score at any single width.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# A small, validated palette: three categorical hues that stay separable under
# deuteranopia and tritanopia.
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e7e6e2"
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a"]

def new_axes(title, xlabel, ylabel, subtitle=None, size=(7.6, 4.3)):
    figure, ax = plt.subplots(figsize=size, facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    ax.set_title(title, color=INK, fontsize=13, fontweight="600", loc="left",
                 pad=20 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.035, subtitle, transform=ax.transAxes, color=INK_MUTED,
                fontsize=9, va="bottom")
    ax.set_xlabel(xlabel, color=INK_MUTED, fontsize=10)
    ax.set_ylabel(ylabel, color=INK_MUTED, fontsize=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9.5, length=0)
    return figure, ax

def label_line_ends(ax, x, entries):
    """Direct labels at the right edge, nudged apart so converging lines stay readable.

    Labels wear the ink token, never the series colour — the mark beside them
    already carries identity.
    """
    low, high = ax.get_ylim()
    minimum_gap = (high - low) * 0.06
    placed = []
    for y, text in sorted(entries, key=lambda entry: entry[0]):
        if placed and y - placed[-1][0] < minimum_gap:
            y = placed[-1][0] + minimum_gap
        placed.append((y, text))
    for y, text in placed:
        ax.annotate(text, (x, y), xytext=(8, 0), textcoords="offset points",
                    color=INK, fontsize=9.5, va="center", annotation_clip=False)

def rate_axis(ax, dims):
    ax.set_xscale("log", base=2)
    ax.set_xticks(dims)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v)}"))
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())

DIMS = report["experiment"]["matryoshka_dims"]
LABELS = {"classification": "Classification", "sts": "STS", "pair": "Pair"}

figure, ax = new_axes(
    "Task quality against embedding width",
    "embedding dimension (log scale)",
    "mean score across tasks",
    subtitle=f"{report['experiment']['method'].upper()} · {report['experiment']['model']}",
)

# Hue carries the task family; a dashed line carries the metric. The families
# are what differ in kind, so they get the categorical slots; macro-F1 is the
# same family measured another way and must not read as a fourth family.
# (docs/MIPIC.pdf reports macro-F1, so both are plotted.)
ends = []
for index, family in enumerate(f for f in LABELS if f in report["summary"]):
    colour = CATEGORICAL[index % len(CATEGORICAL)]
    for metric_key, style, suffix in (
        (family, "-", ""),
        (f"{family}_f1", (0, (5, 2)), " F1"),
    ):
        per_dim = report["summary"].get(metric_key)
        if per_dim is None:
            continue
        values = [per_dim[f"dim_{d}"] for d in DIMS]
        ax.plot(DIMS, values, color=colour, linewidth=2, linestyle=style,
                marker="o", markersize=6, markeredgecolor=SURFACE, markeredgewidth=1.4,
                label=f"{LABELS[family]}{suffix}", zorder=3)
        ends.append((values[-1], f"{LABELS[family]}{suffix}  {values[-1]:.3f}"))

rate_axis(ax, DIMS)
ax.set_xlim(DIMS[0] * 0.85, DIMS[-1] * 2.9)
# Both a legend and direct labels: identity is never carried by colour alone.
label_line_ends(ax, DIMS[-1], ends)
ax.legend(frameon=False, loc="lower right", fontsize=9.5, labelcolor=INK_MUTED)
plt.tight_layout()
plt.show()

## 8. Keep the results

In [ ]:

import shutil

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/embedding-mrl") / RUN_NAME
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(OUTPUT_DIR, destination)
    print("copied to", destination)
else:
    archive = shutil.make_archive(f"/content/{RUN_NAME}", "zip", OUTPUT_DIR)
    print("archived to", archive)
    try:
        from google.colab import files

        files.download(archive)
    except ImportError:
        pass

print("\nrun directory contents:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}  ({path.stat().st_size / 1024:.0f} KB)")